In [ ]:
import docker
import ipaddress
import random

client = docker.from_env()

def get_network_info(network_name):
    """
    获取网络的子网、网关以及已使用的IP地址列表
    """
    try:
        network = client.networks.get(network_name)
        # 刷新属性以获取最新状态
        network.reload() 
    except docker.errors.NotFound:
        print(f"错误: 网络 {network_name} 不存在")
        return None, None, set()

    ipam_config = network.attrs.get('IPAM', {}).get('Config', [])
    if not ipam_config:
        print(f"错误: 网络 {network_name} 没有 IPAM 配置")
        return None, None, set()

    # 通常取第一个配置
    subnet = ipam_config[0].get('Subnet')
    gateway = ipam_config[0].get('Gateway')
    
    # 获取已分配的IP
    used_ips = set()
    if 'Containers' in network.attrs and network.attrs['Containers']:
        for container_id, data in network.attrs['Containers'].items():
            ipv4 = data.get('IPv4Address')
            if ipv4:
                # Docker返回的IP通常带掩码 (e.g., 172.18.0.2/16)，我们需要去掉掩码
                used_ips.add(ipv4.split('/')[0])
    
    # 把网关也算作已使用IP
    if gateway:
        used_ips.add(gateway)

    return subnet, gateway, used_ips

In [ ]:
def find_available_ip(subnet_str, used_ips):
    """
    计算并返回一个可用的IP地址
    """
    network = ipaddress.IPv4Network(subnet_str)
    # hosts() 方法会排除网络地址和广播地址
    possible_hosts = list(network.hosts())
    
    # 简单的随机策略，避免每次都取同一个
    random.shuffle(possible_hosts)
    
    for ip in possible_hosts:
        ip_str = str(ip)
        if ip_str not in used_ips:
            return ip_str
            
    raise Exception("该网段内没有可用的IP地址了")

def add_container_with_route(image_name, network_name, container_name, interface_name="eth0"):
    """
    启动容器，指定IP，并修改默认路由
    """
    # 1. 获取网络信息
    subnet, gateway, used_ips = get_network_info(network_name)
    if not gateway:
        print("无法找到网关，无法继续。")
        return

    # 2. 找到一个可用IP
    try:
        target_ip = find_available_ip(subnet, used_ips)
        print(f"分配到的可用IP: {target_ip} (网关: {gateway})")
    except Exception as e:
        print(e)
        return

    # 3. 启动容器
    # 注意: cap_add=['NET_ADMIN'] 是必须的，否则无法执行 ip route 命令
    try:
        container = client.containers.run(
            image_name,
            name=container_name,
            detach=True,
            network=network_name,
            cap_add=['NET_ADMIN'], 
            command="sleep infinity" # 保持容器运行用于测试，你可以换成你的应用
        )
        
        # 这是一个特殊的步骤：如果容器启动时指定了IP，需要稍作处理。
        # 上面的run命令直接指定network可能会让Docker自动分配IP。
        # 如果要指定静态IP，通常需要先创建容器或使用 networking_config (较为复杂)。
        # 最简单的方法是：先断开默认连接（如果IP不对），或者在连接时指定IP。
        
        # 更稳妥的做法：先创建网络配置，直接在启动时指定IP
        # 由于 docker-py 的 run 方法对 static ip 支持有限，我们改用 connect 方法
        # 这里的逻辑调整为：
        # A. 启动容器 (如果不指定网络，默认用 bridge，或者先设为 none)
        # B. 将容器连接到目标网络并指定 IP
    except Exception as e:
        print(f"容器启动失败: {e}")
        return

    # 修正启动逻辑：为了指定IP，我们重新操作网络连接
    # 此时容器已经连上了 network_name (自动分配了IP)，我们需要断开重连或者直接使用下面的逻辑
    
    # 如果你想严格控制IP，最好的流程是：
    # 1. container = client.containers.create(image_name, network_mode='none', ...)
    # 2. network.connect(container, ipv4_address=target_ip)
    # 3. container.start()
    
    # 让我们用更严谨的"断开-重连"方式来确保IP是我们计算出的那个（针对已存在的容器逻辑类似）
    # 或者直接在 run 的时候不指定 network，利用 connect 指定 IP
    
    container.stop()
    container.remove()
    
    # --- 严谨的启动流程 ---
    print(f"正在启动容器 {container_name} 并绑定 IP {target_ip}...")
    container = client.containers.create(
        image_name,
        name=container_name,
        network_mode='none', # 先不连网络
        cap_add=['NET_ADMIN'],
        command="tail -f /dev/null", # 保持运行
        detach=True
    )
    
    network_obj = client.networks.get(network_name)
    network_obj.connect(container, ipv4_address=target_ip)
    container.start()
    
    # 4. 修改路由
    # Docker 默认会配置一条 default via gateway。
    # 如果你想"手动"加，通常需要先删掉默认的，或者你的仿真环境特殊（无默认路由）。
    
    print(f"正在配置路由: ip route add default via {gateway} dev {interface_name}")
    
    # 这里的 interface_name 在 Docker 中默认通常是 eth0，如果你做了重命名请传入 net0
    exit_code, output = container.exec_run(f"ip route del default") # 先尝试删除现有的默认路由
    
    cmd = f"ip route add default via {gateway} dev {interface_name}"
    exit_code, output = container.exec_run(cmd)
    
    if exit_code == 0:
        print("路由添加成功！")
        # 验证一下
        _, check_out = container.exec_run("ip route show")
        print(f"当前路由表:\n{check_out.decode('utf-8')}")
    else:
        print(f"路由添加失败: {output.decode('utf-8')}")

# --- 使用示例 ---
if __name__ == "__main__":
    # 假设你已经有一个名为 'my-simulation-net' 的 docker network
    # docker network create --subnet=192.168.100.0/24 my-simulation-net
    
    NET_NAME = "my-simulation-net"
    IMG_NAME = "alpine" # 必须安装了 iproute2 (alpine 自带)
    CONT_NAME = "sim-node-01"
    
    # 确保网络存在 (测试用)
    try:
        client.networks.get(NET_NAME)
    except:
        print(f"创建测试网络 {NET_NAME}...")
        client.networks.create(NET_NAME, driver="bridge", ipam=docker.types.IPAMConfig(pool_configs=[docker.types.IPAMPool(subnet='192.168.100.0/24')]))

    add_container_with_route(IMG_NAME, NET_NAME, CONT_NAME, interface_name="eth0")